This notebook is to use a cross-encoder to rerank retrievals for a query.

Collect 100 retrievals (using any retrieval, primarily hybrid) for each question. The cross-encoder reranks these retrievals and chooses the top 10 most relevant ones.

upload corpus.jsonl and hybrid_search_retrievals.csv


In [1]:
!pip install -q transformers
from google.colab import files

In [2]:
!nvidia-smi
import torch; print(torch.cuda.is_available())

Sat Nov  7 15:19:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import ast, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEVICE = "cuda"
tok = AutoTokenizer.from_pretrained("ncbi/MedCPT-Cross-Encoder")
model = AutoModelForSequenceClassification.from_pretrained("ncbi/MedCPT-Cross-Encoder").to(DEVICE).half().eval()

CORPUS = pd.read_json("corpus.jsonl", lines=True, dtype={"pmid": str}).set_index("pmid")
qs = pd.read_csv("hybrid_search_retrievals.csv")
qs["retrievals"] = qs["retrievals"].apply(ast.literal_eval)   # "[1, 2, ...]" text -> real list
print(len(qs), "questions,", len(qs["retrievals"][0]), "candidates each")

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

5729 questions, 100 candidates each


Rerank:

In [4]:
def rerank(question, pmids, k=10, chunk=50):
    docs = CORPUS.loc[[str(p) for p in pmids]]
    texts = [f"{t} {a}" for t, a in zip(docs["title"], docs["abstract"])]
    scores = []
    with torch.no_grad():
        for i in range(0, len(texts), chunk):
            pairs = [[question, t] for t in texts[i:i + chunk]]
            enc = tok(pairs, truncation=True, padding=True, max_length=512, return_tensors="pt").to(DEVICE)
            scores.extend(model(**enc).logits.squeeze(dim=1).float().tolist())
    ranked = sorted(zip(pmids, scores), key=lambda x: -x[1])
    return [p for p, _ in ranked[:k]]

qs["reranked"] = [rerank(q, r) for q, r in tqdm(zip(qs["question"], qs["retrievals"]), total=len(qs))]

Streaming output truncated to the last 5000 lines.
 56%|█████▋    | 3230/5729 [44:07<35:02,  1.19it/s]

 56%|█████▋    | 3231/5729 [44:08<34:37,  1.20it/s]

 56%|█████▋    | 3232/5729 [44:09<34:19,  1.21it/s]

 56%|█████▋    | 3233/5729 [44:10<34:07,  1.22it/s]

 56%|█████▋    | 3234/5729 [44:10<33:05,  1.26it/s]

 56%|█████▋    | 3235/5729 [44:11<33:22,  1.25it/s]

 56%|█████▋    | 3236/5729 [44:12<33:38,  1.23it/s]

 57%|█████▋    | 3237/5729 [44:13<33:56,  1.22it/s]

 57%|█████▋    | 3238/5729 [44:14<33:18,  1.25it/s]

 57%|█████▋    | 3239/5729 [44:15<33:48,  1.23it/s]

 57%|█████▋    | 3240/5729 [44:15<34:40,  1.20it/s]

 57%|█████▋    | 3241/5729 [44:16<35:29,  1.17it/s]

 57%|█████▋    | 3242/5729 [44:17<36:48,  1.13it/s]

 57%|█████▋    | 3243/5729 [44:18<36:24,  1.14it/s]

 57%|█████▋    | 3244/5729 [44:19<35:15,  1.17it/s]

 57%|█████▋    | 3245/5729 [44:20<34:36,  1.20it/s]

 57%|█████▋    | 3246/5729 [44:21<34:23,  1.20it/s]

 57%|█████▋    | 3247/5729 [44:21<34:19,  1.21it

In [8]:
qs[["question", "qid", "type", "reranked"]].to_csv("reranked_top10.csv", index=False)
# files.download("reranked_top10.csv")